# Silver: Skill Extraction

Extracts technical and soft skills from job descriptions using:

## Extraction Methods

1. **Keyword matching** (deterministic, high confidence)
2. **Pattern matching** (regex patterns for skills)
3. **NLP extraction** (spaCy entity recognition)
4. **Future**: LLM-based extraction for complex cases

## Output

- `silver.silver_skill_mapping` table with extracted skills
- Includes: skill name, extraction method, confidence, evidence text
- Skills normalized to taxonomy (future enhancement)

In [0]:
dbutils.widgets.dropdown("extraction_method", "all", ["all", "keyword", "pattern", "nlp"], "Extraction Method")
dbutils.widgets.text("batch_limit", "1000", "Max records to process (empty for all)")

extraction_method = dbutils.widgets.get("extraction_method")
batch_limit = dbutils.widgets.get("batch_limit").strip()

print(f"Extraction Method: {extraction_method}")
print(f"Batch Limit: {batch_limit if batch_limit else 'All records'}")

In [0]:
import json
import re
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"

run_timestamp = F.current_timestamp()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Comprehensive skill dictionary (extendable)
SKILL_KEYWORDS = {
    # Programming Languages
    "python": ["python", "python3", "py"],
    "javascript": ["javascript", "js", "node.js", "nodejs"],
    "java": ["java", "java se", "java ee"],
    "typescript": ["typescript", "ts"],
    "sql": ["sql", "t-sql", "pl/sql", "mysql", "postgresql"],
    "r": [" r ", "r programming"],
    "scala": ["scala"],
    "go": ["golang", " go "],
    "rust": ["rust"],
    "php": ["php"],
    "ruby": ["ruby", "ruby on rails"],
    "c++": ["c++", "cpp"],
    "c#": ["c#", "csharp"],
    
    # Frameworks & Libraries
    "react": ["react", "react.js", "reactjs"],
    "angular": ["angular", "angularjs"],
    "vue": ["vue", "vue.js"],
    "django": ["django"],
    "flask": ["flask"],
    "spring": ["spring", "spring boot"],
    "laravel": ["laravel"],
    "express": ["express", "express.js"],
    
    # Data & ML
    "spark": ["spark", "apache spark", "pyspark"],
    "hadoop": ["hadoop"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "tensorflow": ["tensorflow", "tf"],
    "pytorch": ["pytorch"],
    "scikit-learn": ["scikit-learn", "sklearn"],
    "keras": ["keras"],
    
    # Cloud & DevOps
    "aws": ["aws", "amazon web services"],
    "azure": ["azure", "microsoft azure"],
    "gcp": ["gcp", "google cloud"],
    "docker": ["docker", "containerization"],
    "kubernetes": ["kubernetes", "k8s"],
    "jenkins": ["jenkins"],
    "terraform": ["terraform"],
    "ansible": ["ansible"],
    "ci/cd": ["ci/cd", "continuous integration", "continuous deployment"],
    
    # Databases
    "mongodb": ["mongodb", "mongo"],
    "redis": ["redis"],
    "elasticsearch": ["elasticsearch", "elastic"],
    "cassandra": ["cassandra"],
    "dynamodb": ["dynamodb"],
    
    # Methodologies
    "agile": ["agile", "scrum", "kanban"],
    "devops": ["devops"],
    "machine learning": ["machine learning", "ml"],
    "data science": ["data science"],
    "ai": ["artificial intelligence", "ai"]
}

print(f"Loaded {len(SKILL_KEYWORDS)} skill categories")

In [0]:
# Load active jobs that haven't been processed for skills
jobs_query = f"""
SELECT 
    enterprise_job_id,
    source_name,
    title_raw,
    description_raw,
    company_name_raw
FROM {SILVER_SCHEMA}.silver_jobs_current
WHERE is_active = true 
  AND soft_delete_flag = false
  AND description_raw IS NOT NULL
  AND enterprise_job_id NOT IN (
      SELECT DISTINCT enterprise_job_id 
      FROM {SILVER_SCHEMA}.silver_skill_mapping
  )
"""

if batch_limit:
    jobs_query += f" LIMIT {batch_limit}"

jobs_df = spark.sql(jobs_query)
job_count = jobs_df.count()
print(f"Jobs to process: {job_count}")

if job_count == 0:
    dbutils.notebook.exit({"status": "success", "message": "No new jobs to process"})

In [0]:
def extract_skills_keyword(text, title=""):
    """Extract skills using keyword matching"""
    if not text:
        return []
    
    text_lower = (text + " " + title).lower()
    found_skills = []
    
    for skill_name, keywords in SKILL_KEYWORDS.items():
        for keyword in keywords:
            if keyword in text_lower:
                # Find the actual occurrence for evidence
                match = re.search(r'.{0,30}' + re.escape(keyword) + r'.{0,30}', text_lower, re.IGNORECASE)
                evidence = match.group(0) if match else keyword
                
                found_skills.append({
                    "skill_name_raw": keyword,
                    "skill_name_normalized": skill_name,
                    "extraction_method": "KEYWORD",
                    "evidence_text": evidence,
                    "confidence": 0.95  # High confidence for exact keyword match
                })
                break  # Only match once per skill
    
    return found_skills

# Register as UDF
skill_extract_udf = F.udf(extract_skills_keyword, ArrayType(StructType([
    StructField("skill_name_raw", StringType()),
    StructField("skill_name_normalized", StringType()),
    StructField("extraction_method", StringType()),
    StructField("evidence_text", StringType()),
    StructField("confidence", DoubleType())
])))

print("Keyword extraction function registered")

In [0]:
# Apply skill extraction
skills_extracted_df = jobs_df.withColumn(
    "extracted_skills",
    skill_extract_udf(F.col("description_raw"), F.col("title_raw"))
)

# Explode skills array to individual rows
skills_exploded_df = skills_extracted_df.select(
    F.col("enterprise_job_id"),
    F.col("source_name"),
    F.explode("extracted_skills").alias("skill")
).select(
    F.col("enterprise_job_id"),
    F.col("source_name"),
    F.col("skill.skill_name_raw"),
    F.col("skill.skill_name_normalized"),
    F.col("skill.extraction_method"),
    F.col("skill.evidence_text"),
    F.col("skill.confidence")
)

# Add metadata
skills_final_df = skills_exploded_df.withColumn(
    "skill_mapping_id", F.expr("uuid()")
).withColumn(
    "batch_id", F.lit(run_id)
).withColumn(
    "extracted_at", run_timestamp
)

skills_count = skills_final_df.count()
print(f"Extracted {skills_count} skill mentions from {job_count} jobs")

if skills_count > 0:
    display(skills_final_df.limit(10))

In [0]:
# Create table if not exists
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_skill_mapping (
  skill_mapping_id STRING,
  enterprise_job_id STRING,
  skill_name_raw STRING,
  skill_name_normalized STRING,
  extraction_method STRING,
  evidence_text STRING,
  confidence DECIMAL(5,4),
  batch_id STRING,
  extracted_at TIMESTAMP
)
USING DELTA
""")

# Write extracted skills
if skills_count > 0:
    skills_final_df.select(
        "skill_mapping_id",
        "enterprise_job_id",
        "skill_name_raw",
        "skill_name_normalized",
        "extraction_method",
        "evidence_text",
        F.col("confidence").cast("decimal(5,4)"),
        "batch_id",
        "extracted_at"
    ).write.format("delta").mode("append").saveAsTable(f"{SILVER_SCHEMA}.silver_skill_mapping")
    
    print(f"Wrote {skills_count} skill mappings to table")
else:
    print("No skills extracted")

In [0]:
if skills_count > 0:
    # Top skills extracted
    top_skills_df = spark.sql(f"""
    SELECT 
        skill_name_normalized,
        COUNT(DISTINCT enterprise_job_id) as job_count,
        COUNT(*) as mention_count,
        AVG(confidence) as avg_confidence
    FROM {SILVER_SCHEMA}.silver_skill_mapping
    WHERE batch_id = '{run_id}'
    GROUP BY skill_name_normalized
    ORDER BY job_count DESC
    LIMIT 20
    """)
    
    print("\n=== Top Skills Extracted ===")
    display(top_skills_df)

# Exit summary
result = {
    "status": "success",
    "run_id": run_id,
    "jobs_processed": job_count,
    "skills_extracted": skills_count,
    "avg_skills_per_job": round(skills_count / job_count, 2) if job_count > 0 else 0
}

print("\n=== Skill Extraction Summary ===")
print(json.dumps(result, indent=2))

dbutils.notebook.exit(json.dumps(result))